# Readme E-Manuscripta

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-Manuscripta-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .

Um eine Collection korrekt vorzubereiten, sollten die einzelnen Scripts immer alle in der richtigen Reihenfolge ausgeführt werden.

## 1 - Ordnerstruktur erstellen

Die Scripts in diesem Notebook basieren darauf, dass folgende Ordnerstruktur vorhanden ist, und dass die Ordner jeweils zu Beginn geleert werden.

- Ordner 'files': hier werden diverse Dateien (z.B. eine Textdatei mit allen Signaturen) abgelegt
- Ordner 'info': hier werden die info.json Dateien abgelegt.
- Ordner 'metadata': hier werden die semantischen Metadaten zu den LZA-Objekten abgelegt.     
- Ordner 'objects': hier werden die zu archivierenden Datenobjekte (Payload) abgelegt. Dieser Ordner wird nie geleert, da hier potenziell die Original-Zipkapseln liegen.


In [ ]:
import os
import shutil
from datetime import datetime

# remove existing directories
if os.path.exists('files'):    
    shutil.rmtree('files')

if os.path.exists('info'):
    shutil.rmtree('info')
    
if os.path.exists('metadata'):
    shutil.rmtree('metadata')

# exception: don't remove directory 'objects'
if os.path.exists('objects'):
    pass
else:
    os.mkdir('objects')
 
    
# create necessary directories:
os.mkdir('files')
os.mkdir('info')
os.mkdir('metadata')  

# list all directories in current working directory:
working_directory = './'
print(f"Working directory contains the following directories:")
items = os.listdir(working_directory)
for item in items:
    if os.path.isdir(item):
        print(item)
        
        
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

## 2 - Erstellung der info.json

Dieses Script erstellt eine info.json Datei für alle Objekte der Collection und legt sie im Ordner 'info' als JSON Datei ab.

### Basiskonfiguration mit Config.py

Beispieldaten für ZHB E-Manuscripta. Anpassungen können in der config.py vorgenommen werden. 

    address = 'mailto:someone@internet.com'
    collection = 'ZHB E-Manuscripta'
    collection_id = 'zhb_emanuscripta'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    signature = 'zhb_'
    
Die E-Manuscripta-Signaturen setzen sich aus dem E-Manuscripta DOI zusammen. Im Config-File wird die Abteilung hinzugefügt, das Script ergänzt den DOI. 

### Eingabedatei

Die Excel-Datei liegt im working directory. Sie kann relativ leicht in Alma exportiert werden. Die E-Manuscripta sind in folgendem Set in der RZS gelistet:

    dlza_sosa_e-manuscripta

Die Export-Datei wurde leicht überarbeitet. Nicht benötigte Spalten werden gelöscht, einige Datenmüssen gesplitted werden:

- Signatur aus Spalte Availability splitten, umbenennen zu Call number
- MMS_ID als Text erzwingen
- DOI händisch ergänzen
- Dateipfad händisch ergänzen
- E-Manuscripta ID händisch ergänzen (für IIIF Daten)

Da die Sammlung überschaubar ist, hält sich der zeitliche Aufwand dafür in Grenzen.
Wichtig ist hier vor allem die HAN-Nummer, die in vielen Fällen für die Dateibezeichnungen verwendet wird.


### Export info.json

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory 'files' als menschenlesbarer Nachweis, welche Datenobjekte eingelagert wurden. Auch eine Liste aller Signaturen wird als Text-Datei dort abgelegt.

In [ ]:
import json
import config
import pandas as pd
import requests
from datetime import datetime

# Read the Excel file into a pandas DataFrame

input_file = "e-manuscripta.xlsx"
df = pd.read_excel(input_file)

# other variables:

completeSet = []
today = datetime.today().strftime('%Y-%m-%d')
now = datetime.now().isoformat()

sigfile = "files/signatures.txt"

for _, row in df.iterrows():
    
    doi = row['DOI']
    foldername = doi.replace('.','_').replace('/','_') # folder name structure
    mms_id = str(row['MMS ID'])
    han_id = str(row['Record number'][5:])
    emanuscripta_id = str(row['e-manuscripta ID'])
    print("MMS ID:", mms_id)
    callno = row['Call number']
    doiurl = config.urldoi+doi
    almaurl = config.urlalma+mms_id
    
    infoSet = {
        'additional': foldername,
        'address': config.address, 
        'collection': config.collection,
        'collection_id': config.collection_id,
        'created': now, 
        'identifiers': ['doi:'+doi, 'mmsid:'+mms_id, 'han:'+han_id, 'callnr:'+callno, 'emanuscripta:'+emanuscripta_id],
        'ingest_workflow': config.ingest_workflow, 
        'keywords': config.keywords, 
        'last_changed': now,
        'organisation' : config.organisation,
        'organisation_id' : config.organisation_id,
        'references' : [doiurl, almaurl],
        'signature': config.signature+foldername,
        'sets' : config.sets,
        'title' : row['Title'],
        'user' : config.user      
    }
    
    print(infoSet['identifiers'])
    signature = infoSet['signature']
    print("Signature:",signature)
    completeSet.append(infoSet)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    
    infofile = f"info/{signature}.json"
    with open(infofile, "w") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
    
    # write signature to file
    with open(sigfile, 'a') as file:
        file.write(signature)
        file.write("\n")
        print(f"signatures appended to {sigfile}\n")


# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "files/emanuscripta_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nAll JSON written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "files/emanuscripta_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"All data saved to Excel file as {fullexcelfile}")


## 3 - Semantische Metadaten

### Metadaten aus Alma (SRU, marcxml)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/{signature}/signature.xml. Für gocfl create müssen die Metadaten pro Objekt in einem eigenen Ordner liegen.

### IIIF Manifest aus e-manuscripta.ch

Direkter Download als JSON via e-manuscripta ID. Beispiellink: 

https://www.e-manuscripta.ch/i3f/v20/1040251/manifest


### Weitere Metadaten

Es liegt keine ARK vor. In den ZIP-Kapseln ist jeweils eine METS-Datei vorhanden. Theoretisch  könnten die E-Manuscripta auch via OAI-Abfrage aus der Zenodo-OAI-Schnittstelle abgeholt werden. Sie befinden sich in folgender community: https://zenodo.org/communities/lara_e-manuscripta
Allerdings sind hier die Metadaten rudimentär. 






In [ ]:
import requests
import json
import os
from datetime import datetime
import config

sru = config.urlsru
iiif = config.urliiif

file_name = 'files/emanuscripta_complete_set.json'

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        
        # find necessary values
        alma_id = value["identifiers"][1].split(':')[1]
        foldername = value["additional"]
        signature = value["signature"]
        iiifname = value["identifiers"][4].split(':')[1]
        
        # create new directory for metadata
        os.mkdir(f'metadata/{foldername}')

        # get SRU response
        query = sru+alma_id
        response = requests.get(query)
        if response.status_code != 200:
            raise Exception(f"SRU request failed with status code {response.status_code}")
        
        # Save the response content as xml to a new directory
        metafile = f"metadata/{foldername}/{signature}.xml"
        
        with open(metafile, 'wb') as file:
            file.write(response.content)
            print(f"\nRecord with ID {alma_id} saved as {metafile}\n---")
        
        # get IIIF from e-manuscripta
        query = iiif+iiifname+'/manifest'
        
        response = requests.get(query)
        if response.status_code != 200:
            raise Exception(f"IIIF request failed with status code {response.status_code}")
        
        # Save the response content as xml to a new directory
        metafile = f"metadata/{foldername}/{iiifname}-manifest.json"
        
        with open(metafile, 'wb') as file:
            file.write(response.content)
            print(f"\nRecord with ID {iiifname} saved as {metafile}\n---")

print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))



## 4 - Datenobjekte 

Mit Stand vom Juli 2023 gibt es 22 Objekte in E-Manuscripta der ZHB. Aufgrund der geringen Menge und der diversen Metadatenquellen werden diese E-Manuscripta-Zipkapseln von Hand vom Laufwerk G auf die Workbench verschoben. 

Die E-Manuscripta-Zipkapseln sind alle nach folgender Struktur benannt:

    HAN-Nummer _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiel:

    000218785_20150312T000312_master_ver1.zip

Ausnahme: 

"Der ächte Eidsgenoss, eine wöchtentliche Sittenschrift. Zweyter Jahrgang / hrsg. von Johann Jakob Spreng", https://dx.doi.org/10.7891/e-manuscripta-108732 (keine HAN-Nummer vorhanden => DOI).

Die E-Codices-Objekte liegen auf G:\ZHB-Sosa_Digital\digital unter folgenden Pfaden:

    G:\ZHB-Sosa_Digital\digital\Pp
    G:\ZHB-Sosa_Digital\digital\Ms
    
Die vollständigen Pfade auf G sind in der Eingabedatei ergänzt. 
Der Dateipfad auf der Workbench wird nach dem DOI benannt und ist in die Infojson unter 'additional' abgelegt. 
Dieses Script erstellt lediglich die Unterordner mit dem korrekten Namen, die Files werden händisch hierhin verlegt. 

In [3]:
import os

file_name = 'files/emanuscripta_complete_set.json'

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        foldername = value["additional"]
        
        # make a directory for each object
        
        if os.path.exists(f'objects/{foldername}'):
            pass
        else:
            os.mkdir(f'objects/{foldername}')
            
print("Finished creating folder names at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


Finished creating folder names at  2023-09-13 16:42:20
